In [ ]:
!pip install spacy
!python -m spacy download es_dep_news_trf 
!python -m spacy download es_core_news_sm


In [2]:
import pandas as pd
import spacy

C:\Users\vanes\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nlp = spacy.load('es_core_news_sm')

In [3]:
df1= pd.read_excel('datos_sin procesar/concat/23-10-2022.xlsx')

In [4]:
data=[]
for row in df1.itertuples():
     doc= nlp(df1.at[row.Index, "text"])
     for ent in doc:
          data.append([row.Index,ent])

In [5]:
data= pd.DataFrame(data, columns=['row','text'])
data

,row,text
0,0,Perro
1,0,perdido
2,0,Puerto
3,0,Real
4,0,Cabo
...,...,...
16571,557,la
16572,557,información
16573,557,.
16574,557,Ó


In [23]:
data.to_excel("datos_sin procesar/ner/23-10-2022.xlsx")

In [1]:
#!pip install simpletransformers
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,f1_score,confusion_matrix

In [2]:
import pandas as pd
data = pd.read_excel("datos_sin procesar/ner/23-10-2022.xlsx")
data= data[0:3558]
data

,row,text,tag
0,0,Perro,O
1,0,perdido,O
2,0,Puerto,PLC
3,0,Real,PLC
4,0,Cabo,PLC
...,...,...,...
3553,114,dm,O
3554,114,por,O
3555,114,favor,O
3556,114,compartan,O


In [3]:
data= data.drop(columns='text')

In [5]:
data.drop_duplicates()

,row,tag
0,0,O
2,0,PLC
12,1,O
16,1,PLC
50,2,O
...,...,...
3497,114,O
3503,114,DOG
3507,114,DAT
3533,114,ACC


In [12]:
vals= data.groupby(['tag']).count()
vals

,row
tag,
ACC,85
COL,30
DAT,78
DOG,33
HRS,4
O,2614
PEL,28
PLC,593
RAZ,50


In [5]:
X= data[["row","text"]]
Y =data["tag"]

In [8]:
x_train, x_test, y_train, y_test = train_test_split(X,Y, test_size =0.2)

In [9]:
#building up train data and test data
train_data = pd.DataFrame({"sentence_id":x_train["row"],"words":x_train["text"],"labels":y_train})
test_data = pd.DataFrame({"sentence_id":x_test["row"],"words":x_test["text"],"labels":y_test})

In [254]:
arr=y_test.values.tolist()

In [10]:
train_data

,sentence_id,words,labels
1283,38,casa,O
2762,93,la,PLC
1710,53,se,O
2681,89,Coruna,PLC
3096,103,sus,O
...,...,...,...
2540,85,",",O
2783,94,en,O
692,18,perro,O
638,17,en,O


In [11]:
from simpletransformers.ner import NERModel,NERArgs

In [12]:
label = data["tag"].unique().tolist()
label

['O',
 'PLC',
 'ACC',
 'RAZ',
 'DAT',
 'SEX',
 'STT',
 'COL',
 'PEL',
 'TAM',
 'DOG',
 'HRS']

In [13]:
args = NERArgs()
args.num_train_epochs = 20
args.learning_rate = 1e-4
args.overwrite_output_dir =True
args.train_batch_size = 32
args.eval_batch_size = 32

model = NERModel('bert', 'bert-base-multilingual-cased',labels=label,args =args, use_cuda=False)

Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForTokenClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at 

In [14]:
history=model.train_model(train_data,eval_data = test_data,acc=accuracy_score)

Epoch 20 of 20: 100%|██████████| 20/20 [25:08<00:00, 75.44s/it]


In [15]:
result, model_outputs, preds_list = model.eval_model(test_data) 

Running Evaluation: 100%|██████████| 4/4 [00:26<00:00,  6.74s/it]
C:\Users\vanes\AppData\Local\Programs\Python\Python39\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: PLC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\vanes\AppData\Local\Programs\Python\Python39\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: DAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\vanes\AppData\Local\Programs\Python\Python39\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: RAZ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\vanes\AppData\Local\Programs\Python\Python39\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: ACC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\vanes\AppData\Local\Programs\Python\Python39\lib\site-packages\se

In [247]:
preds_list2=[item for sublist in preds_list for item in sublist]

In [280]:
accuracy = accuracy_score(arr, preds_list2)
f1_score(arr, preds_list2, average='micro')
precision_score(arr, preds_list2, average='micro')

0.6207865168539326

In [54]:
prediction, model_output = model.predict(["Busco a mi perro perdido, se llama Paco, se perdió por la av Losalamos, CDMX"])
prediction

Running Prediction: 100%|██████████| 1/1 [00:00<00:00,  3.71it/s]


[[{'Busco': 'O'},
  {'a': 'O'},
  {'mi': 'O'},
  {'perro': 'O'},
  {'perdido,': 'O'},
  {'se': 'O'},
  {'llama': 'O'},
  {'Paco,': 'PLC'},
  {'se': 'O'},
  {'perdió': 'O'},
  {'por': 'O'},
  {'la': 'PLC'},
  {'av': 'PLC'},
  {'Losalamos,': 'PLC'},
  {'CDMX': 'PLC'}]]